# Weather Pipeline Walkthrough

This notebook demonstrates the complete data pipeline implemented for the Statfinity Data Engineering take-home assessment.

Pipeline stages:

1. **Extract & Load** — Fetch one logical day's weather data and load it into PostgreSQL.
2. **Transform (dbt)** — Build staging and mart models and validate them with dbt tests.
3. **Result Verification** — Query the transformed mart to verify the business-facing output.

The notebook reuses the same ingestion function and dbt commands executed by the Airflow DAG.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /opt/airflow


In [2]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable.*",
    category=UserWarning,
)

## 1. Extract & Load

This stage loads weather data for **one Airflow logical date** into the raw PostgreSQL table. The ingestion is idempotent, so rerunning the same logical date does not create duplicate records.

In [3]:
from ingestion.utils import load_cities
from ingestion.run_ingestion import run_ingestion_for_date

cities = load_cities()

print(f"Configured cities: {len(cities)}")
cities[:3]

Configured cities: 12


[{'name': 'Bangalore', 'latitude': 12.9716, 'longitude': 77.5946},
 {'name': 'Chennai', 'latitude': 13.0827, 'longitude': 80.2707},
 {'name': 'Mumbai', 'latitude': 19.076, 'longitude': 72.8777}]

In [4]:
TARGET_DATE = "2026-09-19"

rows_loaded = run_ingestion_for_date(TARGET_DATE)

print(f"Rows loaded: {rows_loaded}")

Loaded 1 rows for Bangalore on 2026-09-19
Loaded 1 rows for Chennai on 2026-09-19
Loaded 1 rows for Mumbai on 2026-09-19
Loaded 1 rows for Delhi on 2026-09-19
Loaded 1 rows for Hyderabad on 2026-09-19
Loaded 1 rows for Kolkata on 2026-09-19
Loaded 1 rows for Pune on 2026-09-19
Loaded 1 rows for Ahmedabad on 2026-09-19
Loaded 1 rows for Jaipur on 2026-09-19
Loaded 1 rows for Kochi on 2026-09-19
Loaded 1 rows for Guwahati on 2026-09-19
Loaded 1 rows for Srinagar on 2026-09-19

Ingestion completed for 2026-09-19. Total rows: 12
Rows loaded: 12


### Verify raw table

Confirm that rows for the logical date exist in the raw ingestion table.

In [5]:
from ingestion.loader import get_connection
import pandas as pd

conn = get_connection()

pd.read_sql(
    f"""
    SELECT COUNT(*) AS row_count
    FROM raw_weather_daily
    WHERE date = '{TARGET_DATE}'
    """,
    conn,
)

,row_count
0,12


In [6]:
pd.read_sql(
    f"""
    SELECT
        city,
        date,
        temperature_2m_mean,
        precipitation_sum
    FROM raw_weather_daily
    WHERE date = '{TARGET_DATE}'
    ORDER BY city
    LIMIT 10
    """,
    conn,
)

,city,date,temperature_2m_mean,precipitation_sum
0,Ahmedabad,2026-09-19,26.8,8.1
1,Bangalore,2026-09-19,25.7,1.4
2,Chennai,2026-09-19,29.8,9.5
3,Delhi,2026-09-19,28.9,0.0
4,Guwahati,2026-09-19,29.1,0.2
5,Hyderabad,2026-09-19,28.4,1.1
6,Jaipur,2026-09-19,28.2,0.1
7,Kochi,2026-09-19,28.4,5.4
8,Kolkata,2026-09-19,29.8,7.8
9,Mumbai,2026-09-19,27.4,11.2


### Prove rerun safety

Run ingestion again for the same logical date. The row count should remain unchanged because the loader performs an upsert.

In [7]:
run_ingestion_for_date(TARGET_DATE)

pd.read_sql(
    f"""
    SELECT COUNT(*) AS row_count
    FROM raw_weather_daily
    WHERE date = '{TARGET_DATE}'
    """,
    conn,
)

Loaded 1 rows for Bangalore on 2026-09-19
Loaded 1 rows for Chennai on 2026-09-19
Loaded 1 rows for Mumbai on 2026-09-19
Loaded 1 rows for Delhi on 2026-09-19
Loaded 1 rows for Hyderabad on 2026-09-19
Loaded 1 rows for Kolkata on 2026-09-19
Loaded 1 rows for Pune on 2026-09-19
Loaded 1 rows for Ahmedabad on 2026-09-19
Loaded 1 rows for Jaipur on 2026-09-19
Loaded 1 rows for Kochi on 2026-09-19
Loaded 1 rows for Guwahati on 2026-09-19
Loaded 1 rows for Srinagar on 2026-09-19

Ingestion completed for 2026-09-19. Total rows: 12


,row_count
0,12


The row count is unchanged after rerunning ingestion, demonstrating idempotent loading.

## 2. Transform (dbt)

The dbt project transforms the raw weather table into analytics-ready models.

**Model flow**

`raw_weather_daily` → `stg_weather_daily` → `fct_city_daily`

The staging model standardizes types and column names. The mart creates reporting-friendly metrics including temperature range, daylight hours, sunshine hours, precipitation, and rainy-day flags.


In [8]:
import subprocess

result = subprocess.run(
    [
        "bash",
        "-c",
        "cd /opt/airflow/dbt && DBT_PROFILES_DIR=/opt/airflow/dbt dbt run",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
assert result.returncode == 0, result.stderr

18:25:02  Running with dbt=1.8.8
18:25:03  Registered adapter: postgres=1.8.2
18:25:03  Found 2 models, 11 data tests, 1 source, 424 macros
18:25:03  
18:25:03  Concurrency: 4 threads (target='dev')
18:25:03  
18:25:03  1 of 2 START sql view model staging.stg_weather_daily .......................... [RUN]
18:25:03  1 of 2 OK created sql view model staging.stg_weather_daily ..................... [CREATE VIEW in 0.20s]
18:25:03  2 of 2 START sql table model marts.fct_city_daily .............................. [RUN]
18:25:04  2 of 2 OK created sql table model marts.fct_city_daily ......................... [SELECT 41 in 0.11s]
18:25:04  
18:25:04  Finished running 1 view model, 1 table model in 0 hours 0 minutes and 0.54 seconds (0.54s).
18:25:04  
18:25:04  Completed successfully
18:25:04  
18:25:04  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2



In [9]:
result = subprocess.run(
    [
        "bash",
        "-c",
        "cd /opt/airflow/dbt && DBT_PROFILES_DIR=/opt/airflow/dbt dbt test",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
assert result.returncode == 0, result.stderr

18:25:08  Running with dbt=1.8.8
18:25:09  Registered adapter: postgres=1.8.2
18:25:09  Found 2 models, 11 data tests, 1 source, 424 macros
18:25:09  
18:25:09  Concurrency: 4 threads (target='dev')
18:25:09  
18:25:09  1 of 11 START test assert_plausible_weather_values ............................. [RUN]
18:25:09  2 of 11 START test assert_unique_city_date ..................................... [RUN]
18:25:09  3 of 11 START test not_null_fct_city_daily_city ................................ [RUN]
18:25:09  4 of 11 START test not_null_fct_city_daily_precipitation_mm .................... [RUN]
18:25:10  3 of 11 PASS not_null_fct_city_daily_city ...................................... [PASS in 0.18s]
18:25:10  2 of 11 PASS assert_unique_city_date ........................................... [PASS in 0.19s]
18:25:10  4 of 11 PASS not_null_fct_city_daily_precipitation_mm .......................... [PASS in 0.19s]
18:25:10  1 of 11 PASS assert_plausible_weather_values ..........................

### Verify transformed mart

The mart contains one record per city per calendar day with derived business metrics.

In [10]:
pd.read_sql(
    """
    SELECT
        city,
        weather_date,
        temp_min_c,
        temp_max_c,
        temp_range_c,
        precipitation_mm,
        daylight_hours,
        sunshine_hours,
        is_rainy_day
    FROM marts.fct_city_daily
    ORDER BY weather_date DESC, city
    LIMIT 12
    """,
    conn,
)

,city,weather_date,temp_min_c,temp_max_c,temp_range_c,precipitation_mm,daylight_hours,sunshine_hours,is_rainy_day
0,Ahmedabad,2026-09-19,23.6,30.8,7.2,8.1,12.20,11.41,True
1,Bangalore,2026-09-19,21.3,30.5,9.2,1.4,12.16,11.61,True
2,Chennai,2026-09-19,25.7,33.9,8.2,9.5,12.16,7.67,True
3,Delhi,2026-09-19,25.0,32.8,7.8,0.0,12.23,11.18,False
4,Guwahati,2026-09-19,25.3,32.9,7.6,0.2,12.22,11.73,True
5,Hyderabad,2026-09-19,24.9,32.1,7.2,1.1,12.18,11.16,True
6,Jaipur,2026-09-19,23.6,33.3,9.7,0.1,12.22,11.70,True
7,Kochi,2026-09-19,27.0,31.2,4.2,5.4,12.15,7.67,True
8,Kolkata,2026-09-19,27.5,33.6,6.1,7.8,12.20,10.09,True
9,Mumbai,2026-09-19,24.6,30.3,5.7,11.2,12.18,11.35,True


In [11]:
pd.read_sql(
    """
    SELECT COUNT(*) AS total_rows
    FROM marts.fct_city_daily
    """,
    conn,
)

,total_rows
0,41


### Data quality checks

The dbt project validates the transformed data using built-in and custom tests.

- Source `not_null` tests on raw weather data.
- `not_null` tests on staging and mart columns.
- Custom uniqueness test for one record per city per day.
- Custom plausible weather values test covering realistic temperature and precipitation ranges.


## 3. Airflow orchestration

The Airflow DAG orchestrates the pipeline using three tasks:

1. `extract_load`
2. `dbt_run`
3. `dbt_test`

The DAG uses Airflow's logical execution date (`ds`) so scheduled runs and historical backfills execute the same ingestion function without code changes.

This orchestration was verified separately using:

```bash
make airflow-test
```

which executes the DAG for the logical date `2026-09-19`.
```

In [12]:
import subprocess

subprocess.run(
    ["bash", "-c", "cd /opt/airflow && airflow dags list | grep weather_pipeline"],
    text=True,
    check=True,
)

/home/airflow/.local/lib/python3.11/site-packages/airflow/metrics/base_stats_logger.py:22 RemovedInAirflow3Warning: Timer and timing metrics publish in seconds were deprecated. It is enabled by default from Airflow 3 onwards. Enable timer_unit_consistency to publish all the timer and timing metrics in milliseconds.
/home/airflow/.local/lib/python3.11/site-packages/airflow/triggers/base.py:27 RemovedInAirflow3Warning: Timer and timing metrics publish in seconds were deprecated. It is enabled by default from Airflow 3 onwards. Enable timer_unit_consistency to publish all the timer and timing metrics in milliseconds.


weather_pipeline | /opt/airflow/dags/weather_pipeline_dag.py | airflow | True     


CompletedProcess(args=['bash', '-c', 'cd /opt/airflow && airflow dags list | grep weather_pipeline'], returncode=0)

The successful DAG run confirms that ingestion, dbt transformation, and dbt tests execute in sequence for the specified logical date.


In [13]:
conn.close()